# V3 ECG Classifier — TPU/GPU Training

**Run order: Cell 1 → Cell 2 → Cell 3**

| Cell | Runtime | Duration | What it does |
|------|---------|----------|--------------|
| 1 | **GPU/TPU** | ~1 min | Mount Drive + install deps + copy scripts |
| 2 | **GPU/TPU** | ~5 min | Restore data from Drive tars to local SSD |
| 3 | **GPU/TPU** | ~40-90 min (TPU) / ~2-4 hrs (GPU) | Train V3 26-class model |

**Prerequisites (one-time):**
- `MyDrive/EKG/ekg_datasets/chapman.tar` — Chapman dataset
- `MyDrive/EKG/ekg_datasets/ptbxl.tar` — PTB-XL dataset  
- `MyDrive/EKG/ekg_datasets/challenge2021.tar` — Challenge dataset
- `MyDrive/EKG/models/ecg_multilabel_v2.pt` — V2 model (transfer learning source)
- Scripts (`.py` files) sync automatically via Google Drive for Desktop from `MyDrive/EKG/`

**Output:** Best checkpoint saved to `MyDrive/EKG/models/ecg_multilabel_v3_best.pt` during training.

In [ ]:
# =============================================================================
# Cell 1 — GPU/TPU runtime | Mount Drive + install deps + copy scripts
# IMPORTANT: Do NOT import torch_xla here — it locks /dev/vfio/0 and prevents
#            the training script from accessing the TPU.
# =============================================================================
import time, os, subprocess, sys, shutil
t0 = time.time()
print('=== Cell 1: Setup ===')

import torch

# Detect TPU via filesystem — /dev/vfio/0 present only on TPU runtimes.
# Never import torch_xla in the notebook kernel.
if os.path.exists('/dev/vfio/0'):
    accel_type = 'tpu'
    print('TPU runtime detected')
elif torch.cuda.is_available():
    accel_type = 'gpu'
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU or TPU! Go to Runtime > Change runtime type > T4 GPU or TPU v6e > Save')

# Install torch_xla on TPU if missing — check via subprocess, not import
if accel_type == 'tpu':
    check = subprocess.run([sys.executable, '-c', 'import torch_xla'], capture_output=True)
    if check.returncode != 0:
        print('Installing torch_xla...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_xla'], check=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn'], check=True)

SCRIPTS_DIR = '/content/drive/MyDrive/EKG'
os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

NEEDED = ['multilabel_v3.py', 'multilabel_classifier.py',
          'cnn_classifier.py', 'dataset_chapman.py', 'dataset_challenge.py']
missing = []
for script in NEEDED:
    src = f'{SCRIPTS_DIR}/{script}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{script}')
        print(f'  Copied {script}')
    else:
        missing.append(script)
if missing:
    raise FileNotFoundError(f'Missing scripts on Drive: {missing}')

print(f'Accelerator: {accel_type.upper()}')
print(f'=== Cell 1 done ({time.time()-t0:.0f}s) ===')

=== Cell 1: Setup ===
TPU runtime detected
Mounted at /content/drive
  Copied multilabel_v3.py
  Copied multilabel_classifier.py
  Copied cnn_classifier.py
  Copied dataset_chapman.py
  Copied dataset_challenge.py
Accelerator: TPU
=== Cell 1 done (34s) ===


In [3]:
# =============================================================================
# Cell 2 — GPU/TPU runtime | Restore datasets from Drive tars to local SSD
# Safe to re-run: skips steps already done
# Strategy: copy tar from Drive → local /tmp first, then extract.
# Drive FUSE is unreliable for streaming large files directly to tar.
# =============================================================================
import time, os, subprocess, shutil
from pathlib import Path
t0 = time.time()
print('=== Cell 2: Restore data ===')
os.chdir('/content')

DRIVE_DATA   = '/content/drive/MyDrive/EKG/ekg_datasets'
DRIVE_MODELS = '/content/drive/MyDrive/EKG/models'
CHAPMAN_DIR   = '/content/ekg_datasets/chapman'
CHAPMAN_INDEX = '/content/ekg_datasets/chapman_index.csv'
PTBXL_LOCAL   = '/content/ekg_datasets/ptbxl'
CHALLENGE_DIR = '/content/ekg_datasets/challenge2021'
os.makedirs(CHAPMAN_DIR, exist_ok=True)
os.makedirs(PTBXL_LOCAL, exist_ok=True)
os.makedirs(CHALLENGE_DIR, exist_ok=True)

def extract_tar(name, drive_tar, extract_to, check_dir, check_glob, min_count):
    """Copy tar to local disk, extract, clean up."""
    n = len(list(Path(check_dir).rglob(check_glob)))
    if n >= min_count:
        print(f'  {name} already present: {n} files')
        return
    if not os.path.exists(drive_tar):
        raise FileNotFoundError(f'{name} tar not found at {drive_tar}')
    size_gb = os.path.getsize(drive_tar) / 1e9
    local_tar = f'/tmp/{os.path.basename(drive_tar)}'
    print(f'  {name}: copying {size_gb:.1f} GB to local SSD...')
    shutil.copy(drive_tar, local_tar)
    print(f'  {name}: extracting...')
    r = subprocess.run(['tar', 'xf', local_tar, '-C', extract_to],
                       capture_output=True, text=True)
    os.remove(local_tar)
    if r.returncode != 0:
        print(f'  ERROR: {r.stderr}')
        raise RuntimeError(f'{name} extraction failed (exit {r.returncode})')
    n = len(list(Path(check_dir).rglob(check_glob)))
    print(f'  {name}: done — {n} files ({time.time()-t0:.0f}s)')

# Chapman — tar has ekg_datasets/chapman/ prefix → extract to /content
print('[1/5] Chapman')
extract_tar('Chapman', f'{DRIVE_DATA}/chapman.tar',
            '/content', CHAPMAN_DIR, '*.mat', 44000)

# Chapman index
print('[2/5] Chapman index')
if os.path.exists(CHAPMAN_INDEX):
    print('  Already present')
else:
    drive_idx = f'{DRIVE_DATA}/chapman_index.csv'
    if os.path.exists(drive_idx):
        shutil.copy(drive_idx, CHAPMAN_INDEX)
        print('  Restored from Drive')
    else:
        print('  Building...')
        import dataset_chapman
        dataset_chapman.build_chapman_index(CHAPMAN_DIR, CHAPMAN_INDEX)
        print(f'  Done ({time.time()-t0:.0f}s)')

# PTB-XL — tar has ptbxl/ prefix → extract to /content/ekg_datasets
print('[3/5] PTB-XL')
extract_tar('PTB-XL', f'{DRIVE_DATA}/ptbxl.tar',
            '/content/ekg_datasets', PTBXL_LOCAL, '*.dat', 18000)

# V2 model
print('[4/5] V2 model')
V2_MODEL = '/content/models/ecg_multilabel_v2.pt'
if os.path.exists(V2_MODEL):
    print('  Already present')
else:
    drive_v2 = f'{DRIVE_MODELS}/ecg_multilabel_v2.pt'
    if not os.path.exists(drive_v2):
        raise FileNotFoundError(f'ecg_multilabel_v2.pt not found at {drive_v2}')
    shutil.copy(drive_v2, V2_MODEL)
    print(f'  Restored ({os.path.getsize(V2_MODEL)/1e6:.1f} MB)')

# Challenge 2021 — tar has challenge2021/ prefix → extract to /content/ekg_datasets
print('[5/5] Challenge 2021')
tar_path = f'{DRIVE_DATA}/challenge2021.tar'
if os.path.exists(tar_path):
    extract_tar('Challenge', tar_path,
                '/content/ekg_datasets', CHALLENGE_DIR, '*.mat', 50000)
else:
    print('  WARNING: challenge2021.tar not found — training on PTB-XL + Chapman only')

print(f'=== Cell 2 done ({time.time()-t0:.0f}s) ===')

=== Cell 2: Restore data ===
[1/5] Chapman
  Chapman: copying 5.5 GB to local SSD...
  Chapman: extracting...
  Chapman: done — 45152 files (70s)
[2/5] Chapman index
  Already present
[3/5] PTB-XL
  PTB-XL: copying 2.7 GB to local SSD...
  PTB-XL: extracting...
  PTB-XL: done — 21799 files (98s)
[4/5] V2 model
  Restored (6.9 MB)
[5/5] Challenge 2021
  Challenge: copying 7.3 GB to local SSD...
  Challenge: extracting...
  Challenge: done — 54346 files (187s)
=== Cell 2 done (187s) ===


In [4]:
# =============================================================================
# Cell 3 — GPU/TPU runtime | Train V3 26-class model
# GPU (T4): ~2-4 hrs | TPU (v6e): ~40-90 min
# Best checkpoint saved to Drive automatically during training.
# IMPORTANT: Do NOT import torch_xla here — it locks /dev/vfio/0.
# =============================================================================
import time, os, sys, runpy
t0 = time.time()
print('=== Cell 3: V3 Training (26 classes) ===')
os.chdir('/content')

import torch
if os.path.exists('/dev/vfio/0'):
    accel = 'TPU'
elif torch.cuda.is_available():
    accel = f'GPU ({torch.cuda.get_device_name(0)})'
else:
    raise RuntimeError('No GPU or TPU! Run Cell 1 first.')
print(f'Accelerator: {accel}')
print('-' * 60, flush=True)

# Run training in-process (runpy) so the kernel owns the TPU VFIO device.
# subprocess.Popen cannot be used with TPU — subprocesses cannot inherit
# the exclusive VFIO file descriptor that the Colab kernel holds.
sys.argv = ['multilabel_v3.py']
try:
    runpy.run_path('/content/multilabel_v3.py', run_name='__main__')
except SystemExit as e:
    if e.code not in (None, 0):
        print(f'Training exited with code {e.code}')

print('-' * 60, flush=True)
print(f'=== Cell 3 done ({time.time()-t0:.0f}s) ===')
print('Best model saved to MyDrive/EKG/models/ecg_multilabel_v3_best.pt')

=== Cell 3: V3 Training (26 classes) ===
Accelerator: TPU
------------------------------------------------------------

  V3 Multi-Label ECG Classifier  (26 conditions)
  PTB-XL + Chapman + PhysioNet 2021 Challenge
  Device: TPU (xla:0)
  Batch size: 256
  Learning rate: 1.2e-03 (scaled from 3.0e-04)
  Indexing 21799 records for multi-label...
  Kept 18524 records  (skipped: 3275 no-label, 0 no-file)
  Per-class positives:  NORM=9438  PVC=1027  LVH=1751  IMI=1714  ASMI=2007  CLBBB=536  CRBBB=540  LAFB=1622  1AVB=790  ISC_=1260  NDT=1824  IRBBB=1115
Chapman-Shaoxing: 42390 records loaded
Loading Challenge datasets...
  [challenge] georgia         :  10045 kept  (skipped: 238 no-label, 34 no-hea)
  [challenge] cpsc_2018       :   5634 kept  (skipped: 153 no-label, 46 no-hea)
  [challenge] cpsc_2018_extra :   2461 kept  (skipped: 925 no-label, 27 no-hea)
  [challenge] ningbo          :  32702 kept  (skipped: 1969 no-label, 112 no-hea)
  Challenge total: 50842 records  (skipped: 3285 no-la